# Production ML Systems: Comprehensive Guide
## From Model Training to Deployment
**Focus: ML Engineering Best Practices**

---

This notebook covers:
- Data pipeline design
- Model training & validation
- Model evaluation & metrics
- Deployment strategies
- Monitoring & drift detection
- MLOps workflows
- Common pitfalls & solutions

# 1. DATA PIPELINE DESIGN

In [ ]:
"""Production ML starts with robust data pipelines.
Bad data → bad model → bad predictions.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ============= STEP 1: DATA LOADING & VALIDATION =============
def load_and_validate_data(filepath):
    """Load data and perform quality checks."""
    df = pd.read_csv(filepath)
    
    # Data quality checks
    print(f'Shape: {df.shape}')
    print(f'Missing values:\n{df.isnull().sum()}')
    print(f'Duplicates: {df.duplicated().sum()}')
    print(f'Data types:\n{df.dtypes}')
    
    # Check for outliers
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
        if outliers > 0:
            print(f'{col}: {outliers} outliers detected')
    
    return df

# ============= STEP 2: DATA PREPROCESSING =============
def preprocess_data(df, target_col, categorical_cols=None, numeric_cols=None):
    """Comprehensive preprocessing pipeline."""
    df_clean = df.copy()
    
    # Handle missing values
    # Strategy: median for numeric, mode for categorical
    if numeric_cols:
        for col in numeric_cols:
            if df_clean[col].isnull().sum() > 0:
                df_clean[col].fillna(df_clean[col].median(), inplace=True)
    
    if categorical_cols:
        for col in categorical_cols:
            if df_clean[col].isnull().sum() > 0:
                df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    
    # Remove duplicates
    df_clean = df_clean.drop_duplicates()
    
    # Handle categorical features
    le_dict = {}
    if categorical_cols:
        for col in categorical_cols:
            le = LabelEncoder()
            df_clean[col] = le.fit_transform(df_clean[col])
            le_dict[col] = le  # Save encoder for inference
    
    # Separate features and target
    X = df_clean.drop(columns=[target_col])
    y = df_clean[target_col]
    
    return X, y, le_dict

# ============= STEP 3: TRAIN-VAL-TEST SPLIT =============
def create_data_splits(X, y, test_size=0.2, val_size=0.1, random_state=42):
    """Create stratified train/val/test splits.
    
    For classification: use stratify to maintain class distribution
    For regression: use random split
    For time-series: use temporal split (no shuffle)
    """
    # First split: train+val vs test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y  # For classification
    )
    
    # Second split: train vs val
    val_size_adjusted = val_size / (1 - test_size)  # Adjust for first split
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_size_adjusted,
        random_state=random_state,
        stratify=y_temp
    )
    
    print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
    print(f'Train class dist: {y_train.value_counts().to_dict()}')
    print(f'Val class dist: {y_val.value_counts().to_dict()}')
    print(f'Test class dist: {y_test.value_counts().to_dict()}')
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# ============= STEP 4: FEATURE SCALING =============
def scale_features(X_train, X_val, X_test):
    """Scale features using StandardScaler.
    
    IMPORTANT: Fit scaler ONLY on train set, then transform val/test
    (Prevents data leakage)
    """
    scaler = StandardScaler()
    
    # Fit on train only
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Transform val/test (using train statistics)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Convert back to DataFrame (preserve column names)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
    
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

# ============= INTERVIEW TIPS =============
# ✓ Always validate data quality BEFORE training
# ✓ Handle missing values BEFORE train/test split
# ✓ Stratified split for imbalanced datasets
# ✓ Fit scalers ONLY on train set (prevent leakage)
# ✓ Use val set for hyperparameter tuning, test for final eval
# ✓ Check class/feature distribution across splits

# 2. MODEL TRAINING & VALIDATION

In [ ]:
"""Training models with best practices."""

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
import time

# ============= BASELINE MODEL =============
def train_baseline(X_train, y_train):
    """Simple baseline for comparison."""
    baseline = LogisticRegression(max_iter=1000, random_state=42)
    baseline.fit(X_train, y_train)
    return baseline

# ============= HYPERPARAMETER TUNING =============
def hyperparameter_tuning(X_train, y_train, X_val, y_val):
    """GridSearchCV for hyperparameter optimization."""
    
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15],
        'learning_rate': [0.01, 0.1],
        'subsample': [0.8, 1.0]
    }
    
    model = GradientBoostingClassifier(random_state=42)
    
    # GridSearchCV: exhaustive search over param grid
    grid_search = GridSearchCV(
        model,
        param_grid,
        cv=5,  # 5-fold cross-validation
        scoring='roc_auc',  # For imbalanced data
        n_jobs=-1,  # Use all CPU cores
        verbose=1
    )
    
    print('Tuning hyperparameters...')
    grid_search.fit(X_train, y_train)
    
    print(f'Best params: {grid_search.best_params_}')
    print(f'Best CV score: {grid_search.best_score_:.4f}')
    
    return grid_search.best_estimator_

# ============= CROSS VALIDATION =============
def cross_validate_model(model, X, y, cv=5):
    """K-fold cross-validation.
    
    Why: Estimates generalization error more reliably
    """
    scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='roc_auc'
    )
    
    print(f'CV Scores: {scores}')
    print(f'Mean: {scores.mean():.4f} (+/- {scores.std():.4f})')
    
    return scores

# ============= EARLY STOPPING (for neural networks/boosting) =============
def train_with_early_stopping(X_train, y_train, X_val, y_val):
    """Gradient boosting with early stopping.
    
    Stop training when val metric stops improving
    Prevents overfitting
    """
    model = GradientBoostingClassifier(
        n_estimators=1000,  # Max iterations
        random_state=42,
        validation_fraction=0.1,  # Use 10% of train for early stopping
        n_iter_no_change=50  # Stop if no improvement for 50 iterations
    )
    
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    
    print(f'Trained in {elapsed:.2f}s')
    print(f'Iterations: {model.n_estimators_}')
    
    return model

# ============= INTERVIEW TIPS =============
# ✓ Always start with simple baseline (LR, decision tree)
# ✓ Use cross-validation for robust estimates
# ✓ Hyperparameter tune on VAL set, evaluate on TEST set
# ✓ Use early stopping to prevent overfitting
# ✓ Monitor train vs val loss (detect overfitting)
# ✓ Use stratified k-fold for imbalanced data

# 3. MODEL EVALUATION & METRICS

In [ ]:
"""Comprehensive model evaluation for production."""

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, f1_score,
    accuracy_score, precision_score, recall_score,
    mean_squared_error, mean_absolute_error, r2_score
)
import matplotlib.pyplot as plt

# ============= CLASSIFICATION METRICS =============
def evaluate_classification(y_true, y_pred, y_pred_proba=None):
    """Evaluate classification model comprehensively."""
    
    print('=== Classification Metrics ===')
    
    # Basic metrics
    print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'Recall: {recall_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'F1 Score: {f1_score(y_true, y_pred, average="weighted"):.4f}')
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f'\nConfusion Matrix:\n{cm}')
    
    # Detailed report
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred))
    
    # AUC-ROC (for imbalanced data)
    if y_pred_proba is not None:
        auc = roc_auc_score(y_true, y_pred_proba)
        print(f'\nAUC-ROC: {auc:.4f}')
        
        # Precision-Recall Curve
        precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
        print(f'PR-AUC: {np.trapz(precision, recall):.4f}')
    
    return cm

# ============= REGRESSION METRICS =============
def evaluate_regression(y_true, y_pred):
    """Evaluate regression model."""
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print('=== Regression Metrics ===')
    print(f'MSE: {mse:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE: {mae:.4f}')
    print(f'R2 Score: {r2:.4f}')
    
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}

# ============= CLASS IMBALANCE HANDLING =============
def analyze_class_imbalance(y):
    """Analyze and recommend solutions for imbalanced data."""
    
    counts = pd.Series(y).value_counts()
    print('Class Distribution:')
    print(counts)
    print(f'Ratio: {counts.min() / counts.max():.4f}')
    
    if counts.min() / counts.max() < 0.3:
        print('\n⚠️ Imbalanced data detected!')
        print('Solutions:')
        print('- Use stratified split')
        print('- Use class_weight="balanced" in model')
        print('- Use SMOTE (oversampling minority)')
        print('- Use focal loss (penalizes easy examples)')
        print('- Use AUC-ROC instead of accuracy')

# ============= THRESHOLD TUNING =============
def find_optimal_threshold(y_true, y_pred_proba):
    """Find optimal decision threshold.
    
    Default: 0.5
    Imbalanced data: tune based on business metric
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    
    print(f'Default threshold: 0.5')
    print(f'Optimal threshold (F1): {optimal_threshold:.4f}')
    print(f'Max F1 Score: {f1_scores[optimal_idx]:.4f}')
    
    return optimal_threshold

# ============= INTERVIEW TIPS =============
# ✓ Never use accuracy for imbalanced data (use AUC-ROC, F1)
# ✓ Confusion matrix: TP, FP, TN, FN
# ✓ Precision: TP/(TP+FP) - correctness of positive predictions
# ✓ Recall: TP/(TP+FN) - coverage of positive samples
# ✓ F1 = harmonic mean of precision & recall
# ✓ Tune decision threshold based on business requirements
# ✓ Report all metrics, not just accuracy

# 4. MODEL SERVING & INFERENCE

In [ ]:
"""Serving models in production."""

import pickle
import json
from datetime import datetime
import joblib

# ============= MODEL SERIALIZATION =============
def save_model(model, scaler, metadata, filepath):
    """Save model and preprocessing objects."""
    
    artifacts = {
        'model': model,
        'scaler': scaler,
        'metadata': metadata
    }
    
    # Use joblib (better for sklearn models)
    joblib.dump(artifacts, filepath)
    print(f'Model saved to {filepath}')

def load_model(filepath):
    """Load model and preprocessing objects."""
    artifacts = joblib.load(filepath)
    return artifacts['model'], artifacts['scaler'], artifacts['metadata']

# ============= INFERENCE CLASS =============
class ModelPredictor:
    """Production inference wrapper.
    
    Handles:
    - Input validation
    - Preprocessing
    - Prediction
    - Confidence scoring
    - Logging
    """
    
    def __init__(self, model, scaler, metadata):
        self.model = model
        self.scaler = scaler
        self.metadata = metadata
        self.predictions_log = []
    
    def validate_input(self, data):
        """Validate input features."""
        required_cols = self.metadata['feature_names']
        
        if not isinstance(data, dict):
            raise ValueError('Input must be dict')
        
        for col in required_cols:
            if col not in data:
                raise ValueError(f'Missing feature: {col}')
            
            # Type check
            expected_type = self.metadata['feature_types'][col]
            if not isinstance(data[col], expected_type):
                raise ValueError(f'{col} must be {expected_type}')
        
        return True
    
    def preprocess(self, data):
        """Preprocess single sample."""
        required_cols = self.metadata['feature_names']
        X = pd.DataFrame([data])[required_cols]
        X_scaled = self.scaler.transform(X)
        return X_scaled
    
    def predict(self, data, return_proba=False):
        """Make prediction."""
        try:
            # Validate
            self.validate_input(data)
            
            # Preprocess
            X_scaled = self.preprocess(data)
            
            # Predict
            prediction = self.model.predict(X_scaled)[0]
            
            # Get probability
            if return_proba:
                proba = self.model.predict_proba(X_scaled)[0]
                confidence = max(proba)
            else:
                confidence = None
            
            # Log
            self._log_prediction(data, prediction, confidence)
            
            return {
                'prediction': int(prediction),
                'confidence': float(confidence) if confidence else None,
                'timestamp': datetime.now().isoformat()
            }
        
        except Exception as e:
            return {'error': str(e)}
    
    def batch_predict(self, data_list):
        """Predict on multiple samples (more efficient)."""
        predictions = []
        for data in data_list:
            pred = self.predict(data, return_proba=True)
            predictions.append(pred)
        return predictions
    
    def _log_prediction(self, data, prediction, confidence):
        """Log prediction for monitoring."""
        self.predictions_log.append({
            'input': data,
            'prediction': prediction,
            'confidence': confidence,
            'timestamp': datetime.now()
        })

# ============= INTERVIEW TIPS =============
# ✓ Serialize model with scaler (avoid mismatches)
# ✓ Validate inputs before prediction (prevent errors)
# ✓ Return confidence scores (important for ranking, filtering)
# ✓ Log predictions (for monitoring, debugging)
# ✓ Use batch predictions (more efficient than individual)
# ✓ Handle errors gracefully (return error message)

# 5. MODEL DEPLOYMENT STRATEGIES

In [ ]:
"""Different deployment strategies for production."""

"""1. SHADOW DEPLOYMENT
   - New model runs in parallel
   - Predictions logged but not served
   - Validate performance before switching
   
   Code:
   prod_pred = prod_model.predict(X)
   shadow_pred = shadow_model.predict(X)  # Log but don't use
   compare_predictions(prod_pred, shadow_pred)
"""

"""2. CANARY DEPLOYMENT
   - Route small % of traffic to new model
   - Monitor metrics (latency, error rate, accuracy)
   - Gradually increase traffic: 5% -> 25% -> 50% -> 100%
   - Rollback if metrics degrade
   
   Code:
   traffic_split = 0.05  # 5% to new model
   if random.random() < traffic_split:
       pred = new_model.predict(X)
   else:
       pred = prod_model.predict(X)
"""

"""3. BLUE-GREEN DEPLOYMENT
   - Two identical production environments
   - Blue (active), Green (new version)
   - Load balancer switches traffic instantly
   - Zero downtime, instant rollback
   
   Setup:
   - Blue environment: prod_model v1
   - Green environment: prod_model v2
   - Load balancer: points to Blue
   - After validation: switch to Green
   - Issue found: switch back to Blue
"""

"""4. A/B TESTING
   - Run two models simultaneously
   - Split users into groups
   - Measure business metrics (conversion, engagement)
   - Keep winner
   
   Code:
   user_group = hash(user_id) % 100
   if user_group < 50:
       pred = model_a.predict(X)
   else:
       pred = model_b.predict(X)
   log_business_metrics(user_id, pred, outcome)
"""

# ============= DEPLOYMENT CHECKLIST =============
def deployment_checklist():
    """Verification before production deployment."""
    checks = {
        'Model serialization': 'Model saved with scaler, encoders',
        'Input validation': 'Validates all required features',
        'Error handling': 'Graceful error messages',
        'Latency': 'P95 latency < 100ms',
        'Memory': 'Model fits in container memory',
        'Versioning': 'Model version tracked',
        'Monitoring': 'Metrics logged (pred, latency, error)',
        'Rollback plan': 'Fallback to previous model if needed',
        'Data drift': 'Input distribution monitoring',
        'Output validation': 'Predictions in expected range',
    }
    
    for check, detail in checks.items():
        print(f'[ ] {check}: {detail}')

# ============= INTERVIEW TIPS =============
# ✓ Shadow: validate without affecting users
# ✓ Canary: gradual rollout, quick rollback
# ✓ Blue-Green: zero-downtime switching
# ✓ A/B Test: measure business impact
# ✓ Always have rollback plan
# ✓ Monitor metrics closely during deployment

# 6. MONITORING & DRIFT DETECTION

In [ ]:
"""Production monitoring: detect when model degrades."""

from scipy.spatial.distance import js_divergence
from scipy.stats import ks_2samp

# ============= MODEL DRIFT DETECTION =============
class DriftDetector:
    """Monitor model drift in production."""
    
    def __init__(self, train_features, baseline_accuracy=0.95):
        self.train_features = train_features
        self.baseline_accuracy = baseline_accuracy
        self.prediction_buffer = []
        self.feature_stats = self._compute_baseline_stats()
    
    def _compute_baseline_stats(self):
        """Compute baseline statistics from training data."""
        stats = {}
        for col in self.train_features.columns:
            stats[col] = {
                'mean': self.train_features[col].mean(),
                'std': self.train_features[col].std(),
                'min': self.train_features[col].min(),
                'max': self.train_features[col].max(),
                'quantiles': self.train_features[col].quantile([0.25, 0.5, 0.75]).to_dict()
            }
        return stats
    
    def detect_data_drift(self, recent_features, window_size=1000):
        """Detect INPUT distribution drift (data drift).
        
        Methods:
        1. KS Test: compares distributions statistically
        2. Jensen-Shannon: divergence between distributions
        """
        
        drifts = {}
        
        for col in self.train_features.columns:
            baseline = self.train_features[col].values
            current = recent_features[col].values
            
            # KS Test (p-value < 0.05 = significant drift)
            stat, p_value = ks_2samp(baseline, current)
            
            drifts[col] = {
                'ks_statistic': stat,
                'p_value': p_value,
                'drifted': p_value < 0.05
            }
        
        return drifts
    
    def detect_prediction_drift(self, y_true, y_pred, window_size=1000):
        """Detect OUTPUT drift (model predictions change).
        
        Method:
        - Monitor prediction distribution
        - If positive class % changes significantly -> drift
        """
        
        baseline_positive_rate = self.baseline_accuracy
        current_positive_rate = (y_pred == 1).mean()
        
        drift_ratio = abs(current_positive_rate - baseline_positive_rate) / baseline_positive_rate
        
        is_drifted = drift_ratio > 0.1  # 10% change threshold
        
        return {
            'baseline_rate': baseline_positive_rate,
            'current_rate': current_positive_rate,
            'drift_ratio': drift_ratio,
            'is_drifted': is_drifted
        }
    
    def detect_concept_drift(self, y_true, y_pred, window_size=1000):
        """Detect CONCEPT drift (relationship between features & target changes).
        
        Method:
        - Monitor actual accuracy
        - If accuracy drops significantly -> concept drift
        - Trigger retraining
        """
        
        recent_accuracy = accuracy_score(y_true[-window_size:], y_pred[-window_size:])
        
        accuracy_drop = self.baseline_accuracy - recent_accuracy
        is_drifted = accuracy_drop > 0.05  # 5% drop threshold
        
        return {
            'baseline_accuracy': self.baseline_accuracy,
            'recent_accuracy': recent_accuracy,
            'accuracy_drop': accuracy_drop,
            'is_drifted': is_drifted,
            'action': 'RETRAIN' if is_drifted else 'OK'
        }

# ============= PERFORMANCE MONITORING =============
class PerformanceMonitor:
    """Monitor real-time model performance."""
    
    def __init__(self):
        self.metrics = {
            'inference_times': [],
            'errors': [],
            'predictions': []
        }
    
    def log_inference(self, inference_time, prediction, error=None):
        """Log inference metrics."""
        self.metrics['inference_times'].append(inference_time)
        self.metrics['predictions'].append(prediction)
        if error:
            self.metrics['errors'].append(error)
    
    def get_metrics(self):
        """Get current monitoring metrics."""
        times = self.metrics['inference_times']
        
        if not times:
            return None
        
        return {
            'p95_latency': np.percentile(times, 95),
            'p99_latency': np.percentile(times, 99),
            'avg_latency': np.mean(times),
            'error_rate': len(self.metrics['errors']) / len(times),
            'predictions_count': len(self.metrics['predictions'])
        }

# ============= INTERVIEW TIPS =============
# ✓ Data drift: input distribution changed (retrain with new data)
# ✓ Concept drift: relationship changed (retrain model)
# ✓ Prediction drift: model output changed
# ✓ Monitor latency, error rate, accuracy continuously
# ✓ Set thresholds for alerts (accuracy < 0.90)
# ✓ Automate retraining when drift detected

# 7. COMMON PRODUCTION PITFALLS

In [ ]:
"""Common mistakes and how to avoid them."""

# ============= PITFALL 1: DATA LEAKAGE =============
"""Data leakage: information from test/val sneaks into training.

❌ WRONG:
X, y = load_data()
X_scaled = scaler.fit_transform(X)  # Fit on ALL data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)

✓ CORRECT:
X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(X, y)  # Split FIRST
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit ONLY on train
X_test = scaler.transform(X_test)  # Transform test with train stats
"""

# ============= PITFALL 2: TRAIN/TEST MISMATCH =============
"""Model trained on different data distribution than production.

❌ WRONG:
# Train on 2020 data, deploy to 2024
# User behavior changed completely

✓ CORRECT:
# Regularly retrain with recent data
# Monitor data distribution (drift detection)
# Use temporal splits for time-series
train_dates = data[data['date'] < '2024-01-01']
test_dates = data[data['date'] >= '2024-01-01']
"""

# ============= PITFALL 3: IGNORING CLASS IMBALANCE =============
"""Using accuracy on imbalanced data gives misleading results.

❌ WRONG:
# 99% negative, 1% positive
# Predict all negative: accuracy = 99% (useless!)
accuracy = (y_pred == y_true).mean()

✓ CORRECT:
# Use AUC-ROC, F1, Precision-Recall
auc = roc_auc_score(y_true, y_pred_proba)
f1 = f1_score(y_true, y_pred)
# Or use weighted/stratified approaches
from imblearn.over_sampling import SMOTE
smote = SMOTE()
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
"""

# ============= PITFALL 4: OVERFITTING =============
"""Model memorizes train data, fails on new data.

❌ SIGNS:
# Train accuracy: 99%, Val accuracy: 70%
# Huge gap = overfitting

✓ SOLUTIONS:
# 1. Early stopping
model = GradientBoostingClassifier(n_iter_no_change=50)

# 2. Regularization
model = LogisticRegression(C=0.1)  # Smaller C = stronger regularization

# 3. Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5)

# 4. Feature selection (remove noise)
important_features = feature_importances[:top_k]
"""

# ============= PITFALL 5: NOT VERSIONING MODELS =============
"""Can't rollback or compare model versions.

❌ WRONG:
# Save as 'model.pkl'
# Later: which model? old one? new one?

✓ CORRECT:
# Version with timestamp or git commit
filepath = f'models/model_v{version}_{timestamp}.pkl'
# Or use MLflow
mlflow.sklearn.log_model(model, 'model')
# Track: version, metrics, hyperparams, training data
"""

# ============= PITFALL 6: NOT HANDLING MISSING VALUES =============
"""Model crashes on production data with missing values.

❌ WRONG:
# Ignore NaNs during training
# Production data has NaN -> crash

✓ CORRECT:
# Handle BEFORE training
df.fillna(df.median())  # numeric
df.fillna(df.mode()[0])  # categorical
# Or use sklearn.impute.SimpleImputer
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
"""

# ============= PITFALL 7: LATENCY & PERFORMANCE =============
"""Model too slow for production (need < 100ms).

❌ WRONG:
# Complex ensemble: 5 seconds per prediction
# Users wait too long

✓ SOLUTIONS:
# 1. Quantization (FP32 -> INT8)
from sklearn.tree import DecisionTreeClassifier

# 2. Pruning (remove unnecessary features)
important_features = model.feature_importances_ > threshold

# 3. Caching predictions
cache = {}  # input_hash -> prediction

# 4. Batch predictions
# Process multiple at once (parallel)
"""

print('✓ Review these pitfalls before production deployment!')

# 8. MLOps WORKFLOW

In [ ]:
"""Complete MLOps workflow from data to production."""

"""MLOps Pipeline:
| Term           | Purpose                   | When Used         |
| -------------- | ------------------------- | ----------------- |
| Logging        | Save parameters & metrics | During training   |
| Tracking       | Compare experiments       | Model development |
| Model Registry | Version control models    | Before deployment |
| Deployment     | Serve model via API       | Production        |
| Tracing        | Monitor inference calls   | After deployment  |

1. DATA VERSIONING (DVC)
   - Version control for data
   - Track data changes
   - Reproduce results
   
   Code:
   dvc add data/train.csv
   dvc push  # Push to remote storage

2. EXPERIMENT TRACKING (MLflow)
   - Log hyperparams, metrics, artifacts
   - Compare experiments
   - Reproduce best experiment
   
   Code:
   with mlflow.start_run():
       mlflow.log_param('learning_rate', 0.01)
       mlflow.log_metric('accuracy', 0.95)
       mlflow.sklearn.log_model(model, 'model')

3. MODEL REGISTRY
   - Store trained models
   - Version models
   - Stage (Dev -> Staging -> Production)

4. CI/CD PIPELINE
   - Automated testing
   - Model validation
   - Automated deployment

5. MONITORING & ALERTING
   - Track model performance
   - Detect drift
   - Alert on anomalies
   - Trigger retraining
"""

# ============= EXAMPLE MLflow INTEGRATION =============
# pip install mlflow

# import mlflow
# import mlflow.sklearn

# def train_and_log_model(X_train, y_train, X_val, y_val):
#     with mlflow.start_run(experiment_id='exp_001'):
#         # Log params
#         mlflow.log_param('model_type', 'GradientBoosting')
#         mlflow.log_param('n_estimators', 100)
#         mlflow.log_param('learning_rate', 0.1)
#         
#         # Train
#         model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1)
#         model.fit(X_train, y_train)
#         
#         # Log metrics
#         train_acc = model.score(X_train, y_train)
#         val_acc = model.score(X_val, y_val)
#         mlflow.log_metric('train_accuracy', train_acc)
#         mlflow.log_metric('val_accuracy', val_acc)
#         
#         # Log model
#         mlflow.sklearn.log_model(model, 'model')
#         
#         # Log artifacts
#         mlflow.log_artifact('plots/confusion_matrix.png')
#         
#         return model

# View results: mlflow ui

# ============= INTERVIEW TIPS =============
# ✓ Version data (DVC)
# ✓ Track experiments (MLflow, W&B)
# ✓ Store models with metadata (version, metrics, date)
# ✓ CI/CD for automated testing
# ✓ Monitor metrics continuously
# ✓ Automate retraining pipeline
# ✓ Document everything (data schema, features, metrics)

# 9. FEATURE ENGINEERING BEST PRACTICES

In [ ]:
"""Feature engineering for production models."""

# ============= FEATURE IMPORTANCE =============
def analyze_feature_importance(model, feature_names, top_k=10):
    """Understand which features matter."""
    
    # Tree-based models
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    # Linear models
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])
    else:
        return None
    
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(f'Top {top_k} Important Features:')
    print(feature_importance_df.head(top_k))
    
    return feature_importance_df

# ============= FEATURE SELECTION =============
def select_important_features(model, X_train, feature_names, threshold=0.01):
    """Keep only important features (reduce noise, improve latency)."""
    
    importances = model.feature_importances_
    important_indices = np.where(importances > threshold)[0]
    important_features = [feature_names[i] for i in important_indices]
    
    print(f'Original features: {len(feature_names)}')
    print(f'Selected features: {len(important_features)}')
    print(f'Selected: {important_features}')
    
    return important_features

# ============= FEATURE INTERACTION =============
"""Create interactions between features if they have synergy.

Example: house price model
- Feature 1: square footage
- Feature 2: location (0=bad, 1=good)
- Interaction: square_footage * location
  (Good location matters more for large houses)
"""

def create_interactions(X, feature_pairs):
    """Create polynomial features."""
    X_inter = X.copy()
    
    for f1, f2 in feature_pairs:
        X_inter[f'{f1}_x_{f2}'] = X[f1] * X[f2]
    
    return X_inter

# ============= ENCODING CATEGORICAL FEATURES =============
"""Handle categorical features properly."""

def encode_categorical(X, categorical_cols):
    """One-hot encoding vs label encoding."""
    
    # One-hot: for tree-based models (captures non-linearity)
    X_onehot = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    
    # Label encoding: for linear models (smaller feature space)
    X_label = X.copy()
    le_dict = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X_label[col] = le.fit_transform(X_label[col])
        le_dict[col] = le
    
    return X_onehot, X_label, le_dict

# ============= INTERVIEW TIPS =============
# ✓ Start simple: use domain knowledge
# ✓ Remove low-importance features (reduce noise)
# ✓ Create interactions only if they make sense
# ✓ Document features (what they mean, how computed)
# ✓ One-hot for tree models, label for linear
# ✓ Handle missing values consistently

# 10. PRODUCTION READINESS CHECKLIST

In [ ]:
"""Complete checklist before production deployment."""

production_checklist = {
    'Data Pipeline': [
        '[ ] Data validation (missing, outliers, duplicates)',
        '[ ] Feature preprocessing consistent train/production',
        '[ ] Stratified train/val/test split',
        '[ ] No data leakage (scaler fit only on train)',
        '[ ] Handle edge cases (empty input, out-of-range)',
    ],
    'Model Training': [
        '[ ] Baseline model established',
        '[ ] Hyperparameters tuned on validation set',
        '[ ] Cross-validation performed',
        '[ ] Overfitting checked (train vs val gap)',
        '[ ] Class imbalance handled (stratify/SMOTE/weights)',
    ],
    'Model Evaluation': [
        '[ ] Test set evaluation completed',
        '[ ] Appropriate metrics chosen (not just accuracy)',
        '[ ] Confusion matrix analyzed',
        '[ ] Decision threshold tuned if needed',
        '[ ] Model performance documented',
    ],
    'Deployment': [
        '[ ] Model serialized with scaler & encoders',
        '[ ] Input validation implemented',
        '[ ] Error handling & logging',
        '[ ] Prediction latency < 100ms (P95)',
        '[ ] Model versioning setup',
    ],
    'Monitoring': [
        '[ ] Inference latency tracked',
        '[ ] Prediction distribution logged',
        '[ ] Accuracy/metrics monitored on production data',
        '[ ] Data drift detection implemented',
        '[ ] Concept drift detection implemented',
    ],
    'Operations': [
        '[ ] Rollback plan documented',
        '[ ] Fallback model available',
        '[ ] Alerts setup (accuracy drop, latency spike, errors)',
        '[ ] Retraining pipeline automated',
        '[ ] Runbook for troubleshooting created',
    ],
    'Documentation': [
        '[ ] Feature definitions documented',
        '[ ] Model architecture described',
        '[ ] Training data schema documented',
        '[ ] Known limitations noted',
        '[ ] Intended use cases documented',
    ]
}

def print_checklist():
    """Print deployment checklist."""
    for category, items in production_checklist.items():
        print(f'\n{category}:')
        for item in items:
            print(f'  {item}')

print_checklist()

"""KEY METRICS TO TRACK:

Business Metrics:
- Conversion rate
- User satisfaction
- Revenue impact

Model Metrics:
- Accuracy / AUC-ROC / F1
- Precision / Recall
- Confusion matrix

System Metrics:
- P50/P95/P99 latency
- Throughput (predictions/sec)
- Error rate
- CPU/memory usage

Data Quality Metrics:
- Missing values %
- Feature distribution
- Outlier detection

Drift Metrics:
- Data drift (KS test p-value)
- Concept drift (accuracy drop)
- Prediction drift (output distribution change)
"""

print('\n✓ Use this checklist before deployment!')